# **SMS SPAM CLASSIFIER**

Classify a text message as **spam** or **ham** (legitimate). Every model below uses the same stratified split of the UCI SMS Spam Collection.


### **TEAM MEMBERS**

Add the team names here before submission.


### **INTRODUCTION**

Unsolicited SMS still gets through keyword filters, because the wording is short, informal, and often disguised (`fr33`, premium-rate short codes, currency amounts). This notebook trains a ladder of models on the message text alone:

1. **TF-IDF + logistic regression**, the classical baseline
2. **GRU** in PyTorch, the recurrent model
3. **Fine-tuned DistilBERT**, the transformer
4. **LIME and SHAP**, so the spam decision can be traced back to tokens

Spam is the positive class. About 87% of the messages are ham, so accuracy alone is a weak score. The comparison uses spam precision, recall, F1, and precision-recall AUC. Each model picks its spam threshold on the validation set, then that threshold is applied once to the test set.


### **DATASET INFORMATION**

The [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection) has **5,574** English messages: **4,827 ham (86.6%)** and **747 spam (13.4%)**. The working file is `sms_spam.csv` with two columns:

- `Category`: `ham` or `spam`
- `Text`: the raw message

The notebook drops duplicate message strings before splitting. There are no rows whose text is labeled both ham and spam.

Citation: Almeida, Gómez Hidalgo, and Yamakami, “Contributions to the Study of SMS Spam Filtering,” ACM DocEng 2011.


### **IMPORTING NECESSARY PACKAGES**


In [ ]:
from pathlib import Path
import json
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import plotly.express as px
from wordcloud import WordCloud, STOPWORDS

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

if Path("sms_spam.csv").exists():
    CODE_DIR = Path(".").resolve()
elif Path("Code/sms_spam.csv").exists():
    os.chdir("Code")
    CODE_DIR = Path(".").resolve()
else:
    raise FileNotFoundError("Open this notebook from the repository root or from Code/.")

sys.path.insert(0, str(CODE_DIR))
from preprocess import preprocess_text

ARTIFACTS = CODE_DIR / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

SEED = 42
np.random.seed(SEED)

# Skip a stage by setting the flag to False. The comparison table uses whichever models ran.
RUN_RNN = True
RUN_BERT = True
RUN_EXPLAIN = True
RNN_EPOCHS = 5
BERT_EPOCHS = 2

results = []
predictions = {}


### **LOADING THE DATA**


In [ ]:
df = pd.read_csv(CODE_DIR / "sms_spam.csv")
df["Category"] = df["Category"].str.strip().str.lower()
df["Text"] = df["Text"].astype(str)
df["label"] = (df["Category"] == "spam").astype(int)
print("Rows:", len(df))
df.head()


### **EXPLORATORY DATA ANALYSIS**

The checks below match a standard course pass over the file: relevant columns, missing values, class balance, message length, and the words that dominate each class.


In [ ]:
print("Columns:", list(df.columns))
print("Total number of rows in the dataset are", len(df))
print("Missing values:")
print(df[["Category", "Text"]].isnull().sum())
print(df["Category"].value_counts())
df["Length"] = df["Text"].str.len()
df["Length"].describe()


In [ ]:
counts = (
    df["Category"].value_counts().rename_axis("Category").reset_index(name="Messages")
)
fig = px.pie(
    counts,
    names="Category",
    values="Messages",
    hole=0.55,
    title="Spam vs Ham",
    color="Category",
    color_discrete_map={"ham": "#5eead4", "spam": "#2993ae"},
)
fig.update_layout(height=420, width=640)
fig.show()

fig = px.bar(
    counts,
    x="Category",
    y="Messages",
    text="Messages",
    title="Class counts",
    color="Category",
    color_discrete_map={"ham": "#5eead4", "spam": "#2993ae"},
)
fig.update_layout(height=420, width=640, showlegend=False)
fig.show()


## Class distribution: spam vs ham

Ham is about **86.6%** of the collection and spam is about **13.4%**. The set is not balanced. Later models use class weights, and the reported F1 is the spam-class F1 rather than accuracy.


In [ ]:
fig = px.histogram(
    df,
    x="Length",
    color="Category",
    barmode="overlay",
    nbins=40,
    opacity=0.75,
    title="Message length by category",
    color_discrete_map={"ham": "#60a5fa", "spam": "#f59e0b"},
)
fig.update_layout(
    height=420,
    width=800,
    xaxis_title="Characters",
    yaxis_title="Messages",
)
fig.show()
df.groupby("Category")["Length"].describe()


## Distribution of text length

Spam messages tend to be longer and more uniform than ham: a prize notice or a short-code offer fills more of an SMS than a short reply. Length is a useful clue, and it is already inside the raw text the models read.


In [ ]:
def generate_wordcloud(frame, title, colormap, filename):
    text = " ".join(frame["Text"].astype(str))
    cloud = WordCloud(
        stopwords=STOPWORDS,
        max_words=200,
        max_font_size=180,
        width=1400,
        height=600,
        background_color=None,
        mode="RGBA",
        colormap=colormap,
        random_state=SEED,
    ).generate(text)
    plt.figure(figsize=(14, 6))
    plt.imshow(cloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(title, fontsize=18, pad=12)
    plt.tight_layout()
    plt.savefig(CODE_DIR / filename, dpi=200, transparent=True)
    plt.show()

generate_wordcloud(df[df["Category"] == "spam"], "Spam", "plasma", "spam_wordcloud.png")
generate_wordcloud(df[df["Category"] == "ham"], "Ham", "cool", "ham_wordcloud.png")


### **TRAIN / VALIDATION / TEST SPLIT**

Duplicate texts are removed first. No message text is labeled both ham and spam. The remaining rows are split **80% train / 10% validation / 10% test**, stratified on the label, with seed 42. TF-IDF is fit on train only.


In [ ]:
before = len(df)
df = df.drop_duplicates(subset=["Text"]).reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate messages. Rows left: {len(df)}")

train_df, temp_df = train_test_split(
    df, test_size=0.20, stratify=df["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED
)
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    part.to_csv(ARTIFACTS / f"{name}.csv", index=False)
    spam_rate = part["label"].mean()
    print(f"{name:5}  rows={len(part):4}  spam={part['label'].sum():3}  spam_rate={spam_rate:.3f}")

y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()


## **PREPROCESSING**

Classical TF-IDF and the GRU share `preprocess_text` in `preprocess.py`:

1. Lowercasing
2. Redacted spans such as `&lt;#&gt;` and `&lt;URL&gt;` mapped to `num` and `url`
3. Other URLs collapsed to the token `url`
4. Punctuation removed, **digits and £/$ kept** (short codes and prices are spam signal)
5. Extra whitespace removed
6. A small set of function words removed

SMS slang is left as-is (`wif`, `u`, `ur`). Stemming those forms would throw away the difference between a real text and a template. DistilBERT does not use this cleaner. It tokenizes the raw `Text` column.


In [ ]:
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()
for part in (train_df, val_df, test_df):
    part["clean"] = part["Text"].map(preprocess_text)

print("RAW:  ", train_df["Text"].iloc[0])
print("CLEAN:", train_df["clean"].iloc[0])


## **TEXT VECTORIZATION USING TF-IDF**

Two views of each message are concatenated:

- word unigrams and bigrams, top 5,000 features
- character n-grams of length 3–5, top 3,000 features, so spellings like `fr33` still overlap `free`

`FeatureUnion` is fit on the training texts only and saved as `artifacts/vectorizer.pkl`.


In [ ]:
%%time
word_vec = TfidfVectorizer(ngram_range=(1, 2), max_features=5000, min_df=2)
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=3000, min_df=2)
vectorizer = FeatureUnion([("word", word_vec), ("char", char_vec)])

X_train = vectorizer.fit_transform(train_df["clean"])
X_val = vectorizer.transform(val_df["clean"])
X_test = vectorizer.transform(test_df["clean"])
print("X_train", X_train.shape, "X_val", X_val.shape, "X_test", X_test.shape)
joblib.dump(vectorizer, ARTIFACTS / "vectorizer.pkl")


### **MODEL BUILDING AND EVALUATION**

`binary_metrics` scores a probability vector at a chosen threshold. `best_threshold` searches the validation set for the cutoff with the highest spam F1.


In [ ]:
def best_threshold(y_true, proba):
    best_t, best_f = 0.5, -1.0
    for threshold in np.linspace(0.05, 0.95, 19):
        score = f1_score(y_true, (proba >= threshold).astype(int), zero_division=0)
        if score > best_f:
            best_f, best_t = score, float(threshold)
    return best_t


def binary_metrics(y_true, proba, threshold):
    pred = (proba >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "pr_auc": average_precision_score(y_true, proba),
        "threshold": threshold,
    }


def report(name, y_true, proba, threshold):
    scores = binary_metrics(y_true, proba, threshold)
    print(name)
    print("Accuracy score:  {:.3f}".format(scores["accuracy"]))
    print("Precision score: {:.3f}".format(scores["precision"]))
    print("Recall score:    {:.3f}".format(scores["recall"]))
    print("F1 score:        {:.3f}".format(scores["f1"]))
    print("PR-AUC:          {:.3f}".format(scores["pr_auc"]))
    print("Threshold:       {:.2f}".format(scores["threshold"]))
    return scores


#### 1) Logistic regression

Balanced class weights keep the spam class from being ignored. The vectorizer and the classifier are what `app_deploy.py` loads.


In [ ]:
%%time
results = []
predictions = {}
lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)
lr.fit(X_train, y_train)
val_proba = lr.predict_proba(X_val)[:, 1]
test_proba = lr.predict_proba(X_test)[:, 1]
lr_threshold = best_threshold(y_val, val_proba)
scores = report("Logistic Regression", y_test, test_proba, lr_threshold)
scores["model"] = "Logistic Regression"
results.append(scores)
predictions["Logistic Regression"] = (test_proba >= lr_threshold).astype(int)
joblib.dump(lr, ARTIFACTS / "logreg_model.pkl")
thresholds = {"logreg": lr_threshold}


#### 2) GRU

Tokens come from the cleaned training texts. The network is an embedding, a single GRU, and a linear spam logit. The loss up-weights spam. Set `RUN_RNN = False` in the import cell to skip this stage.


In [ ]:
%%time
if RUN_RNN:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, Dataset

    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("GRU device:", device)

    class Vocab:
        def __init__(self, texts, max_size=8000):
            from collections import Counter
            counts = Counter()
            for text in texts:
                counts.update(str(text).split())
            self.pad, self.unk = 0, 1
            self.stoi = {"<pad>": 0, "<unk>": 1}
            for word, _ in counts.most_common(max_size - 2):
                self.stoi[word] = len(self.stoi)

        def encode(self, text, max_len=60):
            ids = [self.stoi.get(tok, self.unk) for tok in str(text).split()][:max_len]
            return ids or [self.unk]

    class SmsDataset(Dataset):
        def __init__(self, texts, labels, vocab):
            self.encoded = [vocab.encode(text) for text in texts]
            self.labels = list(labels)

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, idx):
            return self.encoded[idx], float(self.labels[idx])

    def collate(batch):
        seqs, labels = zip(*batch)
        lengths = torch.tensor([len(seq) for seq in seqs], dtype=torch.long)
        padded = torch.zeros(len(seqs), int(lengths.max()), dtype=torch.long)
        for i, seq in enumerate(seqs):
            padded[i, : len(seq)] = torch.tensor(seq, dtype=torch.long)
        return padded, lengths, torch.tensor(labels, dtype=torch.float32)

    class GRUClassifier(nn.Module):
        def __init__(self, vocab_size, embed_dim=64, hidden=64, pad_idx=0):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
            self.gru = nn.GRU(embed_dim, hidden, batch_first=True)
            self.fc = nn.Linear(hidden, 1)

        def forward(self, tokens, lengths):
            embedded = self.embedding(tokens)
            packed = nn.utils.rnn.pack_padded_sequence(
                embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            _, hidden = self.gru(packed)
            return self.fc(hidden[-1]).squeeze(-1)

    vocab = Vocab(train_df["clean"])
    model_gru = GRUClassifier(len(vocab.stoi)).to(device)
    n_spam = max(int(y_train.sum()), 1)
    n_ham = len(y_train) - int(y_train.sum())
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([n_ham / n_spam], dtype=torch.float32, device=device)
    )
    optimizer = torch.optim.Adam(model_gru.parameters(), lr=1e-3)

    def make_loader(texts, labels, shuffle):
        dataset = SmsDataset(texts, labels, vocab)
        return DataLoader(dataset, batch_size=64, shuffle=shuffle, collate_fn=collate)

    train_loader = make_loader(train_df["clean"], y_train, shuffle=True)

    def gru_proba(texts):
        loader = make_loader(texts, [0] * len(texts), shuffle=False)
        model_gru.eval()
        chunks = []
        with torch.no_grad():
            for tokens, lengths, _ in loader:
                logits = model_gru(tokens.to(device), lengths)
                chunks.append(torch.sigmoid(logits).cpu().numpy())
        return np.concatenate(chunks)

    for epoch in range(1, RNN_EPOCHS + 1):
        model_gru.train()
        total = 0.0
        for tokens, lengths, labels in train_loader:
            tokens = tokens.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model_gru(tokens, lengths), labels)
            loss.backward()
            optimizer.step()
            total += float(loss.item())
        val_f1 = f1_score(y_val, (gru_proba(val_df["clean"]) >= 0.5).astype(int), zero_division=0)
        print(f"epoch {epoch}  train_loss={total / len(train_loader):.4f}  val_f1@0.5={val_f1:.3f}")

    gru_val = gru_proba(val_df["clean"])
    gru_test = gru_proba(test_df["clean"])
    gru_threshold = best_threshold(y_val, gru_val)
    scores = report("GRU", y_test, gru_test, gru_threshold)
    scores["model"] = "GRU"
    results.append(scores)
    predictions["GRU"] = (gru_test >= gru_threshold).astype(int)
    thresholds["gru"] = gru_threshold
    torch.save(
        {
            "state_dict": model_gru.state_dict(),
            "stoi": vocab.stoi,
            "embed_dim": 64,
            "hidden": 64,
        },
        ARTIFACTS / "gru.pt",
    )
else:
    print("Skipping GRU (RUN_RNN is False).")


#### 3) DistilBERT

`distilbert-base-uncased` is fine-tuned on the **raw** message text for `BERT_EPOCHS` epochs (default 2), max length 128, learning rate `2e-5`. The class-weighted loss uses inverse frequency. The checkpoint with the best validation spam F1 is the one that is scored and saved. Set `RUN_BERT = False` to skip this stage. The first run downloads the pretrained weights.


In [ ]:
%%time
if RUN_BERT:
    import copy
    import torch
    from torch import nn
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("DistilBERT device:", device)

    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    bert = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=2
    ).to(device)

    n = len(y_train)
    n_spam = max(int(y_train.sum()), 1)
    n_ham = n - int(y_train.sum())
    class_weights = torch.tensor(
        [n / (2 * n_ham), n / (2 * n_spam)], dtype=torch.float32, device=device
    )
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(bert.parameters(), lr=2e-5)
    train_texts = train_df["Text"].tolist()
    batch_size = 16

    def bert_proba(texts):
        bert.eval()
        chunks = []
        with torch.no_grad():
            for start in range(0, len(texts), 32):
                batch = list(texts[start : start + 32])
                encoded = tokenizer(
                    batch, truncation=True, padding=True, max_length=128, return_tensors="pt"
                )
                encoded = {key: value.to(device) for key, value in encoded.items()}
                probs = torch.softmax(bert(**encoded).logits, dim=-1)[:, 1]
                chunks.append(probs.cpu().numpy())
        return np.concatenate(chunks)

    best_f1, best_state = -1.0, None
    for epoch in range(1, BERT_EPOCHS + 1):
        bert.train()
        order = np.random.permutation(len(train_texts))
        total, steps = 0.0, 0
        for start in range(0, len(order), batch_size):
            idx = order[start : start + batch_size]
            batch_texts = [train_texts[i] for i in idx]
            labels = torch.tensor(y_train[idx], dtype=torch.long, device=device)
            encoded = tokenizer(
                batch_texts, truncation=True, padding=True, max_length=128, return_tensors="pt"
            )
            encoded = {key: value.to(device) for key, value in encoded.items()}
            optimizer.zero_grad()
            loss = criterion(bert(**encoded).logits, labels)
            loss.backward()
            optimizer.step()
            total += float(loss.item())
            steps += 1
        val_f1 = f1_score(y_val, (bert_proba(val_df["Text"]) >= 0.5).astype(int), zero_division=0)
        print(f"epoch {epoch}  train_loss={total / max(steps, 1):.4f}  val_f1@0.5={val_f1:.3f}")
        if val_f1 >= best_f1:
            best_f1 = val_f1
            best_state = copy.deepcopy(bert.state_dict())

    bert.load_state_dict(best_state)
    bert_val = bert_proba(val_df["Text"])
    bert_test = bert_proba(test_df["Text"])
    bert_threshold = best_threshold(y_val, bert_val)
    scores = report("DistilBERT", y_test, bert_test, bert_threshold)
    scores["model"] = "DistilBERT"
    results.append(scores)
    predictions["DistilBERT"] = (bert_test >= bert_threshold).astype(int)
    thresholds["distilbert"] = bert_threshold
    bert.save_pretrained(ARTIFACTS / "distilbert")
    tokenizer.save_pretrained(ARTIFACTS / "distilbert")
else:
    print("Skipping DistilBERT (RUN_BERT is False).")

with open(ARTIFACTS / "threshold.json", "w", encoding="utf-8") as handle:
    json.dump(thresholds, handle, indent=2)


## Model evaluation

The heatmap and confusion matrices use the test set only. Spam is class 1.


In [ ]:
metrics = pd.DataFrame(results).set_index("model")
metrics = metrics[["accuracy", "precision", "recall", "f1", "pr_auc", "threshold"]]
metrics.to_csv(ARTIFACTS / "metrics.csv")
metrics

plt.figure(figsize=(8, 0.8 + 0.6 * len(metrics)))
sns.heatmap(
    metrics[["accuracy", "precision", "recall", "f1", "pr_auc"]],
    annot=True,
    fmt=".2f",
    cmap="Blues",
    cbar=False,
)
plt.title("Metric comparison (test set, spam class except accuracy)")
plt.yticks(rotation=0)
plt.xlabel("Metric")
plt.ylabel("Model")
plt.tight_layout()
plt.show()

names = list(predictions)
fig, axes = plt.subplots(1, len(names), figsize=(5 * len(names), 4))
if len(names) == 1:
    axes = [axes]
for ax, name in zip(axes, names):
    cm = confusion_matrix(y_test, predictions[name], labels=[0, 1])
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=ax,
        xticklabels=["ham", "spam"],
        yticklabels=["ham", "spam"],
    )
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()


## **INTERPRETABILITY**

Three views, on purpose:

- **Coefficients** of the logistic regression: the global spam and ham tokens
- **SHAP** (`LinearExplainer`): how those features move the score on a slice of the test set
- **LIME**: a local explanation of the same individual messages, so the models can be compared side by side

Neural explanations run only for models that were trained in this session.


In [ ]:
feature_names = vectorizer.get_feature_names_out()
coef = lr.coef_.ravel()
top_spam = np.argsort(coef)[-15:][::-1]
top_ham = np.argsort(coef)[:15]
print("Tokens that push the logistic regression toward spam:")
for idx in top_spam:
    print(f"  {coef[idx]:+.3f}  {feature_names[idx]}")
print("Tokens that push it toward ham:")
for idx in top_ham:
    print(f"  {coef[idx]:+.3f}  {feature_names[idx]}")


In [ ]:
if RUN_EXPLAIN:
    import shap

    background = X_train[:400].toarray()
    sample = X_test[:80].toarray()
    explainer = shap.LinearExplainer(lr, background)
    shap_values = explainer.shap_values(sample)
    if isinstance(shap_values, list):
        shap_values = shap_values[-1]
    shap_values = np.asarray(shap_values)
    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, 1]
    mean_abs = np.abs(shap_values).mean(axis=0)
    top_idx = np.argsort(mean_abs)[-15:]
    plt.figure(figsize=(8, 5))
    plt.barh(feature_names[top_idx], mean_abs[top_idx], color="#2993ae")
    plt.title("Mean |SHAP| on a test sample (logistic regression)")
    plt.xlabel("mean |SHAP value|")
    plt.tight_layout()
    plt.show()
else:
    print("Skipping SHAP (RUN_EXPLAIN is False).")


In [ ]:
if RUN_EXPLAIN:
    from lime.lime_text import LimeTextExplainer

    lime_explainer = LimeTextExplainer(class_names=["ham", "spam"])
    lr_pred = predictions["Logistic Regression"]
    test_text = test_df["Text"].to_numpy()

    def take(mask, limit=1):
        hits = np.flatnonzero(mask)
        return hits[:limit]

    chosen = []
    chosen += [("true spam", i) for i in take((y_test == 1) & (lr_pred == 1))]
    chosen += [("true ham", i) for i in take((y_test == 0) & (lr_pred == 0))]
    chosen += [("false positive", i) for i in take((y_test == 0) & (lr_pred == 1))]
    chosen += [("false negative", i) for i in take((y_test == 1) & (lr_pred == 0))]

    def lr_predict_proba(texts):
        cleaned = [preprocess_text(text) for text in texts]
        return lr.predict_proba(vectorizer.transform(cleaned))

    def gru_predict_proba(texts):
        spam = gru_proba([preprocess_text(text) for text in texts])
        return np.column_stack([1.0 - spam, spam])

    def bert_predict_proba(texts):
        spam = bert_proba(list(texts))
        return np.column_stack([1.0 - spam, spam])

    explainers = [("Logistic Regression", lr_predict_proba)]
    if "GRU" in predictions:
        explainers.append(("GRU", gru_predict_proba))
    if "DistilBERT" in predictions:
        explainers.append(("DistilBERT", bert_predict_proba))

    for kind, index in chosen:
        message = test_text[index]
        print("=" * 80)
        print(f"{kind} | gold={'spam' if y_test[index] == 1 else 'ham'}")
        print(message)
        for model_name, predict_fn in explainers:
            samples = 200 if model_name != "Logistic Regression" else 400
            explanation = lime_explainer.explain_instance(
                message, predict_fn, num_features=8, num_samples=samples
            )
            print(f"--- LIME {model_name}")
            for token, weight in explanation.as_list():
                print(f"  {weight:+.3f}  {token}")
else:
    print("Skipping LIME (RUN_EXPLAIN is False).")


### **SAVED ARTIFACTS**

| File | Used by |
| --- | --- |
| `artifacts/vectorizer.pkl` | Streamlit app and the logistic regression |
| `artifacts/logreg_model.pkl` | Streamlit app |
| `artifacts/threshold.json` | Spam cutoff chosen on validation |
| `artifacts/gru.pt` | GRU weights and vocabulary |
| `artifacts/distilbert/` | Fine-tuned DistilBERT and its tokenizer |
| `artifacts/metrics.csv` | Comparison table |
| `artifacts/train.csv`, `val.csv`, `test.csv` | The shared split |

Restart the demo from the repository root after this notebook finishes:

```bash
streamlit run Code/app_deploy.py
```
